# Notebook 5 — Polarization & Network Influence

Companion to **Chapters 16 & 17**. Two ideas in one notebook:

1. **Bounded-confidence opinion dynamics** (Hegselmann–Krause) — when does discussion lead to consensus, and when to polarization?
2. **Network cascade models** (Independent Cascade) and *influence maximisation* via the greedy algorithm of Kempe, Kleinberg & Tardos (2003).

All code lives in `code/social/`; the notebook is a thin orchestration layer.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

from code.social.polarization_abm import run_bounded_confidence
from code.social.network_influence import (
    independent_cascade, expected_reach, greedy_seed_selection
)
np.random.seed(0)

## Part A — Bounded-Confidence Polarization

Each agent updates its opinion to the *mean* of neighbours within $\varepsilon$.
We sweep $\varepsilon$ and look at the resulting opinion distribution.

In [ ]:
for eps in [0.10, 0.20, 0.35]:
    res = run_bounded_confidence(n_agents=100, epsilon=eps, steps=50, seed=0)
    plt.figure(figsize=(5, 2))
    plt.hist(res.final_opinions, bins=20, range=(0, 1))
    plt.title(f'$\\varepsilon={eps}$ → {res.n_clusters} cluster(s)')
    plt.xlabel('opinion'); plt.ylabel('agents')
    plt.show()

**Reflection.** Small $\varepsilon$ → polarization clusters; large $\varepsilon$ → consensus. Which regime does your social-media feed resemble, and why?

## Part B — Cascade Diffusion on a Network

Build a Barabási–Albert scale-free network and run the Independent-Cascade model from three different seed strategies:
- random,
- highest-degree,
- KKT greedy.

In [ ]:
g = nx.barabasi_albert_graph(200, 2, seed=0)
K, P, R = 5, 0.05, 100

random_seeds = list(np.random.default_rng(0).choice(g.nodes, K, replace=False))
degree_seeds = [n for n, _ in sorted(g.degree, key=lambda x: -x[1])[:K]]
greedy_seeds = greedy_seed_selection(g, k=K, p=P, n_runs=30, seed=0)

for name, seeds in {'random': random_seeds, 'top-degree': degree_seeds, 'greedy': greedy_seeds}.items():
    r = expected_reach(g, seeds, p=P, n_runs=R, seed=0)
    print(f'{name:10s} seeds={seeds}  E[reach]={r:5.1f}')

**Reflection.** Greedy beats degree, which beats random. The $(1 - 1/e)$ approximation guarantee comes from submodularity (Kempe, Kleinberg & Tardos, 2003).

## Exercises

1. Add a 10 % *stubborn-truthful* fraction (immune nodes). Replot Part B.
2. Swap the Barabási–Albert graph for an Erdős–Rényi graph at matched density. How does the gap between random and greedy change?
3. Re-run Part A with a *biased* initial opinion distribution (e.g., a 70/30 mix). What happens at small $\varepsilon$?